# Claude 图片理解（Image Understanding）

本 Notebook 演示如何通过 **七牛 AIToken** 平台，使用 **Claude Agent SDK** 的内置 Read 工具读取本地图片，让 Claude 分析图片内容。

## 功能简介

Claude Agent SDK 内置的 Read 工具支持读取图片文件（PNG、JPG 等），Claude 会以多模态方式理解图片内容。

- **本地图片读取**：通过 Read 工具读取本地图片文件，无需手动 Base64 编码
- **Agent 自动调用**：在 prompt 中指示 Claude 读取图片路径，Agent 会自动调用 Read 工具

## 1. 环境准备

安装依赖包，并设置环境变量。

In [ ]:
# 安装 Claude Agent SDK
%pip install claude-agent-sdk -q

In [ ]:
import os
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
)

# 七牛 AIToken 平台配置
QINIU_BASE_URL = "https://api.qnaigc.com"
QINIU_API_KEY = os.environ.get("QINIU_API_KEY", "<your-api-key>")
MODEL = "claude-4.6-sonnet"

os.environ["ANTHROPIC_BASE_URL"] = QINIU_BASE_URL
os.environ["ANTHROPIC_API_KEY"] = QINIU_API_KEY
os.environ["ANTHROPIC_MODEL"] = MODEL

# 示例图片路径（与本 Notebook 同目录下的 image.png）
IMAGE_PATH = os.path.join(os.getcwd(), "image.png")
assert os.path.exists(IMAGE_PATH), f"图片不存在: {IMAGE_PATH}"

print("环境配置完成!")
print(f"  API 端点: {QINIU_BASE_URL}")
print(f"  模型: {MODEL}")
print(f"  图片路径: {IMAGE_PATH}")

## 2. 图片理解

使用 Claude Agent SDK 的 `query()` 函数，通过内置 Read 工具读取本地图片。

关键配置：
- `allowed_tools=["Read"]`：允许 Agent 使用 Read 工具读取文件
- `permission_mode="bypassPermissions"`：跳过权限确认，自动执行工具调用
- 在 prompt 中指定图片的绝对路径，Agent 会自动调用 Read 工具读取图片

In [ ]:
# 使用 Claude Agent SDK 读取本地图片并分析
async def image_understanding():
    async for message in query(
        prompt=f"Read the image at {IMAGE_PATH} and 请详细描述这张图片的内容，包括主要元素、颜色、构图和整体氛围。",
        options=ClaudeAgentOptions(
            allowed_tools=["Read"],
            permission_mode="bypassPermissions",
        ),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "text") and block.text.strip():
                    print(f"Claude: {block.text}")
                elif hasattr(block, "name"):
                    print(f"[工具调用: {block.name}]")
        elif isinstance(message, ResultMessage):
            if message.result:
                print(f"\n=== 最终结果 ===")
                print(message.result[:1000])
            print(f"\n--- 执行信息 ---")
            print(f"耗时: {message.duration_ms}ms")
            print(f"费用: ${message.total_cost_usd:.4f}")

await image_understanding()

## 3. 参数说明

### ClaudeAgentOptions 关键参数

| 参数 | 值 | 说明 |
|------|------|------|
| `allowed_tools` | `["Read"]` | 允许 Agent 使用 Read 工具，Read 工具支持读取图片文件 |
| `permission_mode` | `"bypassPermissions"` | 自动执行工具调用，无需人工确认 |

### Read 工具支持的图片格式

- JPEG (`*.jpg`, `*.jpeg`)
- PNG (`*.png`)
- GIF (`*.gif`)
- WebP (`*.webp`)

### 注意事项

1. 图片路径必须是**绝对路径**
2. Read 工具读取图片后，Claude 以多模态方式理解图片内容，无需手动 Base64 编码
3. `permission_mode="bypassPermissions"` 仅在受信任的场景下使用，生产环境建议使用默认权限模式

## 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [Claude Agent SDK Python 文档](https://github.com/anthropics/claude-agent-sdk-python)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)